# 💧 LFM3-Audio - Speech Capabilities

This notebook walks through what [`LiquidAI/LFM3-Audio-2B`](https://huggingface.co/LiquidAI/LFM3-Audio-2B) can do with a few lines of 🤗 Transformers code.
LFM3-Audio is a single end-to-end speech model: it **listens** to audio, **reads** text, and **answers in text, speech, or interleaved text and speech**.

## 🎯 What You'll Find:
- **Speech recognition (ASR)** - transcribe speech in many languages
- **Text-to-speech (TTS)** - read text aloud with a female or male voice in 19 languages and accents
- **Speech translation** - speech to text
- **Spoken chat** - talk to the model and get an answer in interleaved text and speech
- **Speech-to-text chat** - ask by voice, get a text answer, with or without thinking
- **Tool calling** - turn a spoken request into a function call

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type` (an L4 or A100 is recommended, the model runs in bfloat16)
- **Hugging Face Account**: For accessing models and datasets

# 📦 Installation & Setup

First, let's install all the required packages:

In [ ]:
!pip install -qqq "transformers>=5.20.2" accelerate datasets librosa soundfile --progress-bar off

Let's now verify the packages are installed correctly

In [ ]:
import torch
import transformers

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"🖥️ GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'none'}")

# Loading the model from Transformers 🤗

Following the [model card](https://huggingface.co/LiquidAI/LFM3-Audio-2B#transformers-quick-start),
load the model and processor with `trust_remote_code=True`. The processor automatically loads the **Liquid codec**
bundled in `audio_tokenizer/`; access it as `processor.audio_tokenizer` and move it to the model's device.
We use bfloat16 on GPU and float32 on CPU. The codec turns generated speech codes into a 24 kHz waveform.

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Audio-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

print("🧠 Loading model...")
model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=dtype, device_map=device, trust_remote_code=True
).eval()

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device).eval()

print("\n✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

## 🛠️ A small helper

Every task below is a chat conversation. The **system prompt** describes the task. **Text output is the default**:
generation starts in text mode and switches to speech or interleaved output when an output-mode token is encountered.

To select a mode from the start, we **prefill** the assistant turn: put the prefix below in a final assistant message
and let the model continue it. The helper sets `continue_final_message=True` and `add_generation_prompt=False` when
`prefill` is provided, so generation continues that message instead of starting a new assistant turn.

| Assistant prefill | Behavior |
|---|---|
| None (`prefill=None`) | Start in the default text mode; the model can still emit a token that switches modes. |
| `<\|audio_start\|>` | Start speech-only output, for example TTS, until an end-of-audio frame. |
| `<\|mixed_start\|>` | Start interleaved text and speech, alternating blocks of 6 text tokens and 12 speech frames. |
| `<think>` | Begin a thinking block; the answer follows its closing `</think>`. |
| `<think></think>` | Supply an empty thinking block so the answer starts immediately. |

The thinking prefixes control text-only response. For text-only chat, the examples also set `suppress_mixed=True` and `suppress_audio_out=True` to prevent a switch to speech. In interleaved output, `<|text_end|>` closes the text portion while speech continues until an end-of-audio frame.

See the [LFM3-Audio model documentation](https://huggingface.co/LiquidAI/LFM3-Audio-2B#usage-examples)
for the output-mode tokens.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE_OUT = 24_000  # the codec decodes to 24 kHz


@torch.inference_mode()
def generate(conversation, prefill=None, max_new_tokens=512, **generate_kwargs):
    """Runs one assistant turn. Returns (text, waveform or None)."""
    if prefill is not None:
        conversation = conversation + [{"role": "assistant", "content": prefill}]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=prefill is None,
        continue_final_message=prefill is not None,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, **generate_kwargs)
    text = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    waveform = None
    if outputs.audio_codes is not None and outputs.audio_lengths[0] > 0:
        codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
        waveform = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
    return text, waveform


SPEECH_SAMPLING = dict(
    audio_temperature=1.0, 
    audio_top_k=128, 
    audio_repetition_penalty=1.1, 
    audio_repetition_window=4096,
)


def play(waveform, sr=SAMPLE_RATE_OUT):
    display(Audio(waveform, rate=sr))

# 🎙️ Speech recognition (ASR)

Use the system prompt `Perform ASR in <language>.` with the English name of the language, or `Perform ASR.` to let the model detect it.

Audio is passed in the user turn as `{"type": "audio", "audio": ...}`, where the value is a file path or URL. The processor resamples it to 16 kHz mono.
Let's grab a clip from LibriSpeech.

In [ ]:
import io
from datasets import Audio as AudioFeature, load_dataset

librispeech = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
librispeech = librispeech.cast_column("audio", AudioFeature(decode=False))

sample = librispeech[0]
speech, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
sf.write("librispeech.wav", speech, sr)

print("📝 Reference:", sample["text"])
play(speech, sr)

In [ ]:
transcript, _ = generate([
    {"role": "system", "content": "Perform ASR in English."},
    {"role": "user", "content": [{"type": "audio", "audio": "librispeech.wav"}]},
])
print("🎙️ Transcript:", transcript)

## 🎙️ Live ASR

Reuse `librispeech.wav` from the ASR example to simulate microphone input in 100 ms pieces.
[LiveASR](https://huggingface.co/LiquidAI/LFM3-Audio-2B#usage-examples) currently supports English only.
With `Perform live ASR.`, the model emits text for roughly every 0.96 seconds of audio.
Keep both the feature-extractor cache and the transcription state between chunks, then flush at the end.
For streams longer than 30 seconds, pass `input_ids=live_prompt` again to reset the text context while retaining the state and encoder cache.

In [ ]:
import librosa

tokenizer = processor.tokenizer
chunk_frames = 8 * model.config.interleaved_n_audio
live_prompt = tokenizer(
    "<|startoftext|><|im_start|>system\nPerform live ASR.<|im_end|>\n<|im_start|>assistant\n",
    add_special_tokens=False,
    return_tensors="pt",
).input_ids.to(device)
suppress = tokenizer.convert_tokens_to_ids(["<|mixed_end|>", "<|endoftext|>"])
stop = tokenizer.convert_tokens_to_ids(["<|endoftext|>", "<|im_start|>"])
live_state, feature_cache = None, None
mel = torch.empty((1, processor.feature_extractor.feature_size, 0), device=device)


@torch.inference_mode()
def transcribe_chunk(frames, is_final=False):
    global live_state
    output = model.transcribe_live(
        frames,
        live_state,
        input_ids=live_prompt if live_state is None else None,
        is_final=is_final,
        max_new_tokens=96 if is_final else 48,
        suppress_token_ids=suppress,
        stop_token_ids=stop,
    )
    live_state = output.state
    print(tokenizer.decode(output.sequences[0], skip_special_tokens=True), end="", flush=True)


live_speech, _ = librosa.load("librispeech.wav", sr=16_000)
print("🎙️ Live transcript: ", end="", flush=True)
for start in range(0, len(live_speech), 1_600):
    features, feature_cache = processor.feature_extractor.stream(
        live_speech[start:start + 1_600], feature_cache
    )
    mel = torch.cat((mel, features.input_features.to(device)), dim=-1)
    while mel.shape[-1] >= chunk_frames:
        transcribe_chunk(mel[..., :chunk_frames])
        mel = mel[..., chunk_frames:]

# Flush the feature extractor and the model, including any final partial chunk.
features, feature_cache = processor.feature_extractor.stream([], feature_cache, is_final=True)
mel = torch.cat((mel, features.input_features.to(device)), dim=-1)
while mel.shape[-1] > chunk_frames:
    transcribe_chunk(mel[..., :chunk_frames])
    mel = mel[..., chunk_frames:]
transcribe_chunk(mel, is_final=True)
print()

# 🗣️ Text-to-speech (TTS)

Use the system prompt `Perform TTS in <language> using a <female|male> voice.` and prefill `<|audio_start|>`.
Supported: Arabic, Danish, German, Greek, British English, American English, Finnish, French, Hebrew, Italian, Japanese,
Korean, Dutch, Polish, Portuguese, Spanish, Swedish, Turkish and Chinese.

In [ ]:
torch.manual_seed(0)
_, tts_en = generate(
    [
        {"role": "system", "content": "Perform TTS in American English using a female voice."},
        {"role": "user", "content": "I made a little plan for today: go for a walk, find a new workout playlist, and try a new cake recipe"},
    ],
    prefill="<|audio_start|>",
    **SPEECH_SAMPLING,
)
sf.write("tts_en.wav", tts_en, SAMPLE_RATE_OUT)
play(tts_en)

In [ ]:
_, tts_fr = generate(
    [
        {"role": "system", "content": "Perform TTS in French using a female voice."},
        {"role": "user", "content": "On pourrait inventer une nouvelle recette ce soir. Si le résultat est étrange, on lui donnera un nom très chic."},
    ],
    prefill="<|audio_start|>",
    **SPEECH_SAMPLING,
)
sf.write("tts_fr.wav", tts_fr, SAMPLE_RATE_OUT)
play(tts_fr)

A quick sanity check: let the model transcribe its own French speech.

In [ ]:
transcript, _ = generate([
    {"role": "system", "content": "Perform ASR in French."},
    {"role": "user", "content": [{"type": "audio", "audio": "tts_fr.wav"}]},
])
print("🎙️ Transcript:", transcript)

# 🌍 Speech translation

Translation works best into or out of English. The system prompt picks the direction and the output modality:

| Direction | System prompt |
|---|---|
| Speech → text | `Convert the speech in {source} to written {target}.` |

In [ ]:
s2t_translation, _ = generate([
    {"role": "system", "content": "Convert the speech in French to written English."},
    {"role": "user", "content": [{"type": "audio", "audio": "tts_fr.wav"}]},
])
print("🌍 Speech → text:", s2t_translation)

# 💬 Spoken chat (interleaved text and speech)

With the system prompt `Respond with interleaved text and audio.` and the `<|mixed_start|>` prefill, the model answers
conversationally in text and speech at the same time, so speech can start playing before the full answer is written.

Let's ask how to find an enjoyable activity and make it a habit. First, synthesize the question with a male voice.

In [ ]:
_, question = generate(
    [
        {"role": "system", "content": "Perform TTS in American English using a male voice."},
        {"role": "user", "content": "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit?"},
    ],
    prefill="<|audio_start|>",
    **SPEECH_SAMPLING,
)
sf.write("question.wav", question, SAMPLE_RATE_OUT)
play(question)

In [ ]:
reply, reply_speech = generate(
    [
        {"role": "system", "content": "Respond with interleaved text and audio."},
        {"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]},
    ],
    prefill="<|mixed_start|>",
    max_new_tokens=1024,
    **SPEECH_SAMPLING,
)
print("💬 Text:", reply)
play(reply_speech)

# 🧠 Speech-to-text chat

In this mode, the model behaves like a familiar text LLM that also accepts speech. No system prompt is needed.
Prefill `<think>` to let the model reason first; it closes the reasoning with `</think>`, after which comes the answer.
We pass `suppress_mixed=True, suppress_audio_out=True` to keep the answer in text.

In [ ]:
text, _ = generate(
    [{"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]}],
    prefill="<think>",
    max_new_tokens=2048,
    suppress_mixed=True,
    suppress_audio_out=True,
)
thinking, _, answer = text.partition("</think>")
print("🤔 Thinking:", thinking.strip()[:500], "...")
print("\n💡 Answer:", answer.strip())

To skip thinking, prefill an empty `<think></think>` block so the assistant starts answering immediately.

In [ ]:
answer, _ = generate(
    [{"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]}],
    prefill="<think></think>",
    max_new_tokens=1024,
    suppress_mixed=True,
    suppress_audio_out=True,
)
print("💡 Answer:", answer.strip())

Text and audio can be mixed in the same user turn, e.g. a recording plus a written instruction about it:

In [ ]:
answer, _ = generate(
    [{"role": "user", "content": [
        {"type": "audio", "audio": "librispeech.wav"},
        {"type": "text", "text": "Who or what is the speaker talking about, and what is their attitude towards it?"},
    ]}],
    prefill="<think></think>",
    max_new_tokens=512,
    suppress_mixed=True,
    suppress_audio_out=True,
)
print("💡 Answer:", answer.strip())

# 🔧 Tool calling: switching subscriptions

One spoken request asks for two actions: cancel a streaming-platform subscription at the end of the billing period,
then subscribe to a podcast platform's **Max plan with monthly billing**. The model should produce a cancellation
call followed by a subscription call, preserving the platform, plan, billing frequency, and cancellation timing.

Give the tool list in the system prompt, between `<tools>` and `</tools>`, one JSON object per line.
The user asks by voice; the model returns the requested `<tool_call>` blocks.

In [ ]:
import json

tools = [
    {
        "name": "cancel_subscription",
        "description": "Cancel the signed-in user's subscription to a streaming platform at the specified time.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string"},
                "effective_when": {"type": "string"},
            },
            "required": ["platform", "effective_when"],
        },
    },
    {
        "name": "subscribe_to_podcast_platform",
        "description": "Subscribe the signed-in user to a podcast platform with the chosen plan and billing frequency.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string"},
                "plan": {"type": "string"},
                "billing_frequency": {"type": "string"},
            },
            "required": ["platform", "plan", "billing_frequency"],
        },
    },
]
tool_prompt = (
    "Respond with function calls. Use the tools to help the user complete their goal.\n"
    "Current date: 2026-10-27\n"
    "List of tools:\n<tools>\n" + "\n".join(json.dumps(t) for t in tools) + "\n</tools>"
)

_, request = generate(
    [
        {"role": "system", "content": "Perform TTS in British English using a female voice."},
        {"role": "user", "content": "Please cancel my StreamBox subscription starting December first. Then subscribe me to PodSpace on the Max plan, billed monthly."},
    ],
    prefill="<|audio_start|>",
    **SPEECH_SAMPLING,
)
sf.write("request.wav", request, SAMPLE_RATE_OUT)
play(request)

call, _ = generate(
    [
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": [{"type": "audio", "audio": "request.wav"}]},
    ],
    max_new_tokens=512,
    suppress_mixed=True,
    suppress_audio_out=True,
)
print("🔧 Proposed subscription calls (not executed):\n", call)

## 🔧 Function calling with interleaved text and audio

Reuse the tools and `request.wav` above, changing the system instruction to
`Respond with function calls, and interleaved text and audio.`.
As in the [model-card example](https://huggingface.co/LiquidAI/LFM3-Audio-2B#usage-examples), leave `prefill=None`
so the model can choose `<tool_call>` or `<|mixed_start|>`. Keep speech output enabled.

The cell prints each response and plays any generated speech. It parses the proposed calls and returns
**mock results** so the model can demonstrate a spoken follow-up. No subscriptions are changed.
The parser below is scoped to the two example tools, whose parameters are all strings.

In [ ]:
import re


interleaved_tool_prompt = (
    "Respond with function calls, and interleaved text and audio. Use the tools to help the user complete their goal.\n"
    "Current date: 2026-10-27\n"
    "List of tools:\n<tools>\n" + "\n".join(json.dumps(t) for t in tools) + "\n</tools>"
)
tool_conversation = [
    {"role": "system", "content": interleaved_tool_prompt},
    {"role": "user", "content": [{"type": "audio", "audio": "request.wav"}]},
]
call_pattern = re.compile(
    r"<tool_call>\s*<function=([^>\n]+)>(.*?)</function>\s*</tool_call>", re.DOTALL
)
parameter_pattern = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.DOTALL)
tool_schemas = {tool["name"]: tool["parameters"] for tool in tools}

for turn in range(3):
    tool_text, tool_speech = generate(
        tool_conversation,
        prefill=None,
        max_new_tokens=1024,
        **SPEECH_SAMPLING,
    )
    print("🔧 Assistant:", tool_text)
    if tool_speech is not None:
        play(tool_speech)

    calls, results = [], []
    for name, body in call_pattern.findall(tool_text):
        arguments = {key: value.strip() for key, value in parameter_pattern.findall(body)}
        schema = tool_schemas.get(name)
        if schema is None or set(arguments) != set(schema["required"]):
            raise ValueError(f"Unexpected tool call: {name}({arguments})")
        calls.append({"type": "function", "function": {"name": name, "arguments": arguments}})
        results.append({"name": name, "status": "success", "mock": True, **arguments})
    if not calls:
        if "<tool_call>" in tool_text:
            raise ValueError("Incomplete tool call; increase max_new_tokens and inspect the output.")
        break

    print("🧪 Mock tool results:", results)
    tool_conversation.extend([
        {"role": "assistant", "tool_calls": calls},
        {"role": "tool", "content": json.dumps(results)},
    ])
else:
    print("Stopped after three turns; inspect the proposed calls above.")

# 🚀 Next steps

- Fine-tune LFM3-Audio on your own data: see [`lfm3_audio_tts_sft_with_transformers.ipynb`](../finetuning/notebooks/lfm3_audio_tts_sft_with_transformers.ipynb), which teaches the model a new voice.
- Explore LFM3-Omni, which adds image and video understanding on top of everything above: [`lfm3_omni_capabilities.ipynb`](lfm3_omni_capabilities.ipynb).
- Documentation: [docs.liquid.ai](https://docs.liquid.ai/lfm) · Community: [Discord](https://discord.com/invite/liquid-ai)